<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# 11. Comparing Text Chunking Strategies for RAG
**Industry:** Medical Equipments

Implement and compare 4 different text chunking strategies on the same manual and evaluate them.

In [1]:
!pip install langchain langchain-experimental chromadb sentence-transformers langchain-community pandas tabulate

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.11
    Uninstalling langchain-text-splitters-0.3.11:
      Successfully uninstalled langchain-text-splitters-0.3.11
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.3.31
    Uninstalling langchain-community-0.3.31:
      Successfully uninstalled langchain-community-0.3.31


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-firestore 0.5.0 requires langchain-core<1.0.0,>=0.1.1, but you have langchain-core 1.5.3 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import pandas as pd
import os

with open('manual.txt', 'w') as f:
    f.write("The MRI scanner uses magnetic fields. It must be cooled with liquid helium at all times to maintain superconductivity.\n\nMaintenance requires checking helium levels daily using the dipstick tool. Do not bring metal objects near the machine as they will become projectiles.\n\nEmergency shutoff is located on the wall to the left of the console. Press the red button in case of fire or a quenching event.\n\nA quench happens when the liquid helium boils off rapidly, releasing gas into the room. Evacuate immediately if this occurs.")

loader = TextLoader('manual.txt')
docs = loader.load()

embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 1. Fixed-size chunking
fixed = CharacterTextSplitter(chunk_size=100, chunk_overlap=0, separator=" ").split_documents(docs)
# 2. Recursive chunking
recursive = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20).split_documents(docs)
# 3. Semantic chunking
semantic = SemanticChunker(embedder).split_documents(docs)
# 4. Sliding Window (implemented via Recursive with large overlap)
sliding = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=80).split_documents(docs)

collections = {
    "Fixed": Chroma.from_documents(fixed, embedder, collection_name="fixed"),
    "Recursive": Chroma.from_documents(recursive, embedder, collection_name="recursive"),
    "Semantic": Chroma.from_documents(semantic, embedder, collection_name="semantic"),
    "Sliding": Chroma.from_documents(sliding, embedder, collection_name="sliding"),
}

questions = ["Where is the emergency shutoff?", "What happens during a quench?"]

results = []
for q in questions:
    for strategy_name, db in collections.items():
        retrieved = db.similarity_search(q, k=1)[0].page_content
        results.append({"Question": q, "Strategy": strategy_name, "Top Chunk": retrieved})

df = pd.DataFrame(results)
print(df.to_markdown(index=False))

C:\Users\Kaizen\AppData\Local\Temp\ipykernel_16880\1236934717.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "D:\Internship\Teach-ai\Backend\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.